In [ ]:
!pip install fastapi uvicorn transformers torch sse-starlette

# 04 - Streaming Chat API

## Overview

This notebook builds a production-style streaming chat API that serves an LLM via
Server-Sent Events (SSE). We cover model loading, streaming generation, FastAPI server
construction, error handling, logging, and Docker deployment.

**Learning Goals:**
- Load a small LLM (Qwen2.5-0.5B or mock) for streaming
- Implement streaming generation with `TextStreamer` / custom async loop
- Build a FastAPI app with `POST /chat/stream` returning SSE
- Add error handling, request logging, and validation
- Test with Python `httpx` streaming client and `curl`
- Produce a Docker deployment snippet

---

In [ ]:
import torch
import logging
import sys
import time
import json
import asyncio
from typing import AsyncGenerator, Optional

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
logger = logging.getLogger(__name__)
logger.info("Notebook started")

## 1. Load Qwen2.5-0.5B (or Mock for CPU)

We use `Qwen/Qwen2.5-0.5B` -- small enough for CPU. The same patterns apply to any
HuggingFace model. If the model is unavailable, a mock generator is used.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, TextIteratorStreamer
import threading

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

model = None
tokenizer = None
MOCK_MODE = False

logger.info(f"Loading tokenizer: {MODEL_NAME}")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    logger.info(f"Loading model: {MODEL_NAME} on {DEVICE}")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=DTYPE,
        trust_remote_code=True,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    if DEVICE == "cpu":
        model = model.to(DEVICE)
    model.eval()

    param_m = sum(p.numel() for p in model.parameters()) / 1e6
    logger.info(f"Model loaded: {param_m:.1f}M params, vocab={len(tokenizer)}")
    print(f"Model ready: {param_m:.1f}M params on {DEVICE}")

except Exception as e:
    logger.warning(f"Model load failed: {e}. Using MOCK mode.")
    MOCK_MODE = True
    print(f"MOCK MODE: Model unavailable ({e})")
    print("Using mock generator for demonstration.")

## 2. Implement Streaming Generation

Streaming sends tokens as they are generated. We implement three approaches:
1. **TextStreamer** -- synchronous, prints to console
2. **TextIteratorStreamer** -- threaded, yields tokens for API use
3. **Custom async generator** -- manual next-token loop (full control)

In [ ]:
# === 2a. Mock generator (fallback for when model is unavailable) ===

async def mock_generator(prompt: str, max_tokens: int = 50) -> AsyncGenerator[str, None]:
    """Simulate streaming token generation."""
    mock_responses = [
        "Sure! ", "Let ", "me ", "think ", "about ", "that. ",
        "Here ", "is ", "a ", "response: ",
        f"You ", f"asked ", f"'\"{prompt[:20]}...\"' ",
        "and ", "I ", "am ", "generating ", "tokens ",
        "one ", "by ", "one ", "as ", "they ", "arrive. ",
        "[EOS]"
    ]
    count = 0
    for token in mock_responses:
        if count >= max_tokens:
            break
        yield token
        count += 1
        await asyncio.sleep(0.05)


# === 2b. Real async token generator ===

async def generate_tokens_async(
    prompt: str,
    max_new_tokens: int = 128,
    temperature: float = 0.7,
) -> AsyncGenerator[str, None]:
    """
    Generate tokens one at a time using a manual next-token loop.
    Gives full control over each generation step.
    """
    if MOCK_MODE or model is None:
        async for token in mock_generator(prompt, max_new_tokens):
            yield token
        return

    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, 'chat_template') and tokenizer.chat_template:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = prompt

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    past_key_values = None
    generated = 0

    with torch.no_grad():
        while generated < max_new_tokens:
            if past_key_values is None:
                outputs = model(input_ids, use_cache=True)
            else:
                outputs = model(input_ids[:, -1:], past_key_values=past_key_values, use_cache=True)

            logits = outputs.logits[:, -1, :] / temperature
            past_key_values = outputs.past_key_values

            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

            if next_id.item() == tokenizer.eos_token_id:
                break

            token_text = tokenizer.decode(next_id[0], skip_special_tokens=True)
            if token_text:
                yield token_text

            input_ids = next_id
            generated += 1
            await asyncio.sleep(0)


# === 2c. Quick test ===
async def test_stream():
    print(">>> Streaming test:")
    async for token in generate_tokens_async("What is 2+2?", max_new_tokens=30):
        print(token, end="", flush=True)
    print()

await test_stream()

## 3. Build FastAPI App with POST /chat/stream (SSE)

Server-Sent Events (SSE) is the standard protocol for streaming text from a server.
Each token is sent as a `data:` line, and `data: [DONE]` signals completion.

In [ ]:
# === Write the server file ===

SERVER_CODE = r'''
"""streaming_chat_server.py -- FastAPI SSE streaming chat API."""

import asyncio
import json
import logging
import sys
import time
from contextlib import asynccontextmanager
from typing import Optional

import torch
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse, JSONResponse
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
logger = logging.getLogger("chat_api")

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

model: Optional[AutoModelForCausalLM] = None
tokenizer: Optional[AutoTokenizer] = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    global model, tokenizer
    logger.info(f"Loading {MODEL_NAME} on {DEVICE}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=DTYPE, trust_remote_code=True,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    if DEVICE == "cpu":
        model = model.to(DEVICE)
    model.eval()
    param_m = sum(p.numel() for p in model.parameters()) / 1e6
    logger.info(f"Model ready ({param_m:.1f}M params on {DEVICE})")
    yield
    logger.info("Shutting down...")


app = FastAPI(title="Streaming Chat API", version="0.1.0", lifespan=lifespan)


class Message(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    messages: list[Message]
    max_tokens: int = Field(default=128, ge=1, le=1024)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    stream: bool = True


async def generate_sse_stream(messages: list[dict], max_tokens: int, temperature: float):
    """Yield SSE-formatted token events."""
    try:
        if hasattr(tokenizer, "chat_template") and tokenizer.chat_template:
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        else:
            prompt = messages[-1]["content"]

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_ids = inputs["input_ids"]
        past_key_values = None
        generated = 0

        rid = hex(int(time.time() * 1e6))[-8:]
        yield f"data: {json.dumps({'id': rid, 'object': 'chat.completion.chunk', 'choices': [{'delta': {'role': 'assistant'}, 'index': 0}]})}\n\n"

        with torch.no_grad():
            while generated < max_tokens:
                if past_key_values is None:
                    outputs = model(input_ids, use_cache=True)
                else:
                    outputs = model(input_ids[:, -1:], past_key_values=past_key_values, use_cache=True)

                logits = outputs.logits[:, -1, :] / temperature
                past_key_values = outputs.past_key_values

                probs = torch.softmax(logits, dim=-1)
                next_id = torch.multinomial(probs, num_samples=1)

                if next_id.item() == tokenizer.eos_token_id:
                    break

                token_text = tokenizer.decode(next_id[0], skip_special_tokens=True)
                if token_text:
                    chunk = {"id": rid, "object": "chat.completion.chunk",
                             "choices": [{"delta": {"content": token_text}, "index": 0}]}
                    yield f"data: {json.dumps(chunk, ensure_ascii=False)}\n\n"

                input_ids = next_id
                generated += 1
                await asyncio.sleep(0)

        yield "data: [DONE]\n\n"

    except Exception as e:
        logger.exception("Generation error")
        yield f"data: {json.dumps({'error': {'message': str(e), 'type': 'generation_error'}})}\n\n"


@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_NAME, "device": DEVICE}


@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    """Streaming chat endpoint returning SSE."""
    if model is None or tokenizer is None:
        raise HTTPException(status_code=503, detail="Model not loaded")

    logger.info("Request: %d messages, max_tokens=%d", len(req.messages), req.max_tokens)

    if not req.messages:
        raise HTTPException(status_code=400, detail="messages list is empty")

    messages_dicts = [m.model_dump() for m in req.messages]

    return StreamingResponse(
        generate_sse_stream(messages_dicts, req.max_tokens, req.temperature),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
            "X-Accel-Buffering": "no",
        },
    )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")
'''

from pathlib import Path
server_path = Path("/tmp/streaming_chat_server.py")
server_path.write_text(SERVER_CODE)
print(f"Server code written to {server_path}")
print("To start: python /tmp/streaming_chat_server.py")

## 4. Add Error Handling

Production APIs must handle model-not-loaded, invalid inputs, and generation failures gracefully.

In [ ]:
# === Error handling and validation helpers ===

ERROR_HANDLING_CODE = r'''
# Add these to streaming_chat_server.py for production robustness

from fastapi import Request
from fastapi.responses import JSONResponse


# -- Custom exception handlers --
@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={"error": {"message": exc.detail, "type": "http_error", "code": exc.status_code}},
    )


@app.exception_handler(Exception)
async def general_exception_handler(request: Request, exc: Exception):
    logger.exception("Unhandled exception on %s %s", request.method, request.url.path)
    return JSONResponse(
        status_code=500,
        content={"error": {"message": "Internal server error", "type": "internal_error"}},
    )


# -- Input validation --
VALID_ROLES = {"system", "user", "assistant", "function", "tool"}
MAX_CONTENT_LENGTH = 32_000


def validate_messages(messages: list[dict]) -> None:
    if not messages:
        raise HTTPException(status_code=400, detail="messages list is empty")
    total_chars = 0
    for i, msg in enumerate(messages):
        if not isinstance(msg, dict):
            raise HTTPException(status_code=400, detail=f"message[{i}] must be a dict")
        if msg.get("role") not in VALID_ROLES:
            raise HTTPException(status_code=400,
                detail=f"message[{i}] invalid role: {msg.get('role')!r}")
        total_chars += len(str(msg.get("content", "")))
    if total_chars > MAX_CONTENT_LENGTH:
        raise HTTPException(status_code=400,
            detail=f"Total content too long: {total_chars} chars (max {MAX_CONTENT_LENGTH})")
'''

print("Error handling patterns (add to server file):")
print(ERROR_HANDLING_CODE)

In [ ]:
# === Validation test ===

VALID_ROLES = {"system", "user", "assistant", "function", "tool"}

def validate_messages(messages: list[dict]) -> None:
    """Validate chat messages before processing."""
    if not messages:
        raise ValueError("messages list is empty")
    total_chars = 0
    for i, msg in enumerate(messages):
        if msg.get("role") not in VALID_ROLES:
            raise ValueError(f"message[{i}] invalid role: {msg.get('role')!r}")
        total_chars += len(str(msg.get("content", "")))
    if total_chars > 32_000:
        raise ValueError(f"Total content too long: {total_chars} chars")


# Test
for test_case, expect_pass in [
    ([], False),
    ([{"role": "invalid", "content": "hi"}], False),
    ([{"role": "user", "content": "Hello"}], True),
]:
    try:
        validate_messages(test_case)
        result = "PASS" if expect_pass else "UNEXPECTED PASS"
    except ValueError as e:
        result = "FAIL (expected)" if not expect_pass else f"UNEXPECTED FAIL: {e}"
    print(f"  {test_case!r:50s} -> {result}")

## 5. Add Request Logging Middleware

Structured logging helps debug production issues. We log method, path, status, and duration.

In [ ]:
LOGGING_MIDDLEWARE_CODE = r'''
# Add to streaming_chat_server.py

import time
from starlette.middleware.base import BaseHTTPMiddleware
from fastapi import Request


class LoggingMiddleware(BaseHTTPMiddleware):
    """Log every request with method, path, status, and duration."""

    async def dispatch(self, request: Request, call_next):
        start = time.perf_counter()
        response = await call_next(request)
        elapsed_ms = (time.perf_counter() - start) * 1000
        logger.info(
            "%s %s -> %d (%.1fms)",
            request.method, request.url.path, response.status_code, elapsed_ms,
        )
        return response


# Register the middleware:
# app.add_middleware(LoggingMiddleware)
'''

print("Request logging middleware:")
print(LOGGING_MIDDLEWARE_CODE)

## 6. Test with Python httpx Streaming Client

Run the server in another terminal, then test with `httpx` for async streaming.

In [ ]:
# === Test with httpx streaming ===
# Requires: pip install httpx
# Run the server first: python /tmp/streaming_chat_server.py

TEST_CODE_HTTPX = r'''
import httpx
import json


async def test_httpx_streaming(base_url: str = "http://localhost:8000"):
    """Test the streaming endpoint with httpx."""
    url = f"{base_url}/chat/stream"
    payload = {
        "messages": [{"role": "user", "content": "Say hello in 3 languages."}],
        "max_tokens": 60,
        "temperature": 0.7,
        "stream": True,
    }

    print(f"POST {url}")
    collected = []

    async with httpx.AsyncClient(timeout=60.0) as client:
        async with client.stream("POST", url, json=payload) as response:
            print(f"Status: {response.status_code}")
            print(f"Content-Type: {response.headers.get('content-type')}")
            print("\n--- Streaming tokens ---")

            async for line in response.aiter_lines():
                if line.startswith("data: "):
                    data_str = line[6:]
                    if data_str == "[DONE]":
                        print("\n[DONE]")
                        break
                    try:
                        data = json.loads(data_str)
                        token = data["choices"][0]["delta"].get("content", "")
                        if token:
                            print(token, end="", flush=True)
                            collected.append(token)
                    except json.JSONDecodeError:
                        pass

    full = "".join(collected)
    print(f"\n\nFull response ({len(full)} chars): {full}")
    return full


# Run it:
# import asyncio
# asyncio.run(test_httpx_streaming())
'''

print("httpx streaming test client (save as test_client.py):")
print(TEST_CODE_HTTPX)

# Also write it to a file
Path("/tmp/test_client.py").write_text(TEST_CODE_HTTPX)
print("\nAlso written to /tmp/test_client.py")

## 7. curl Command for Manual Testing

Quick one-liner to test without writing code.

In [ ]:
# === curl test commands ===

print("=== Health Check ===")
print("curl http://localhost:8000/health")

print("\n=== Streaming Chat (with -N for no buffering) ===")
print(r'''curl -N -X POST http://localhost:8000/chat/stream \
  -H "Content-Type: application/json" \
  -d '{
    "messages": [{"role": "user", "content": "What is 2+2?"}],
    "max_tokens": 50,
    "temperature": 0.7,
    "stream": true
  }'
''')

print("\n=== With httpie (prettier output) ===")
print(r"""http --stream POST :8000/chat/stream \
  messages:='[{"role":"user","content":"Hi"}]' \
  max_tokens:=50 stream:=true
""")

print("\nNote: Add the -N flag to curl to disable buffering (required for streaming).")

## 8. Docker Snippet for Easy Deployment

Containerize the API for reproducible deployment.

In [ ]:
# === Dockerfile ===

DOCKERFILE = r'''FROM python:3.11-slim

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends curl \
    && rm -rf /var/lib/apt/lists/*

RUN pip install --no-cache-dir \
    torch --index-url https://download.pytorch.org/whl/cpu \
    transformers \
    fastapi \
    uvicorn[standard] \
    pydantic

COPY streaming_chat_server.py .

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=10s --start-period=120s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "streaming_chat_server:app", "--host", "0.0.0.0", "--port", "8000"]
'''

Path("/tmp/Dockerfile").write_text(DOCKERFILE)

# === docker-compose.yml ===

DOCKER_COMPOSE = r'''version: "3.9"

services:
  chat-api:
    build:
      context: .
      dockerfile: Dockerfile
    image: streaming-chat-api:latest
    container_name: chat-api
    ports:
      - "8000:8000"
    volumes:
      - huggingface_cache:/root/.cache/huggingface
    environment:
      - MODEL_NAME=Qwen/Qwen2.5-0.5B
      - DEVICE=cpu
      - MAX_NEW_TOKENS=256
      - LOG_LEVEL=INFO
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 120s

  # Optional nginx reverse proxy
  nginx:
    image: nginx:alpine
    container_name: chat-nginx
    ports:
      - "443:443"
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf:ro
    depends_on:
      - chat-api
    restart: unless-stopped

volumes:
  huggingface_cache:
    driver: local
'''

Path("/tmp/docker-compose.yml").write_text(DOCKER_COMPOSE)

print("Docker files written to /tmp/:")
print("  /tmp/Dockerfile")
print("  /tmp/docker-compose.yml")
print("\nTo deploy:")
print("  cp /tmp/streaming_chat_server.py /tmp/Dockerfile /tmp/docker-compose.yml .")
print("  docker compose up -d")
print("  curl http://localhost:8000/health")

## 9. TODO: Exercises

1. **Non-streaming fallback**: Add a non-streaming response path when `stream=false`.
2. **Rate limiter**: Implement a token-bucket rate limiter as FastAPI middleware.
3. **Model list endpoint**: Add an OpenAI-compatible `/v1/models` endpoint.
4. **Concurrent stress test**: Use `aiohttp` to send 10 concurrent requests and measure latency.
5. **GPU deployment**: Modify the Dockerfile for NVIDIA GPU support (`nvidia/cuda` base image).
6. **Prompt caching**: Cache the KV cache for system prompts to speed up multi-turn conversations.
7. **Stream cancellation**: Handle client disconnection mid-stream gracefully.

## Summary

In this notebook, we built a complete streaming LLM API from scratch:

| Component | What We Built |
|-----------|--------------|
| **Model Loading** | AutoModelForCausalLM with CUDA/CPU detection and mock fallback |
| **Streaming Generation** | Custom async generator with manual next-token loop and KV-cache |
| **FastAPI Server** | Lifespan management, `POST /chat/stream` with SSE output |
| **Error Handling** | HTTP exception handlers, input validation, model-not-loaded guard |
| **Request Logging** | Middleware logging method, path, status, and latency per request |
| **Client Testing** | httpx async streaming client + curl one-liners |
| **Docker Deployment** | Dockerfile + docker-compose with health checks and model cache volume |

**Next:** Move on to the core modules of the LLM Learning Roadmap.

---
*(c) LLM Learning Roadmap - Prerequisites Module*